Connected to Python 3.12.10

In [ ]:
import pandas as pd
import os
print(os.getcwd())

script_dir = os.path.dirname(os.path.abspath(__file__))
print(script_dir)

c:\Users\Wu\College_App_2028\SciFi_20260616
c:\Users\Wu\College_App_2028\SciFi_20260616


In [ ]:
"""in the above example, we directly apply pre-trained RoBERTa the validation dataset.

Now, fine-tune RoBERTa on the training dataset and then evaluate it on the validation datset. 

two different model names ---  
r-roberta-base-sentiment-latest → a pretrained sentiment model (analogy: a finished cake)

roberta-base → a general RoBERTa model you fine-tune yourself (analogy: raw incredients (flour, eggs, sugar)) 

"""
#1. load dataset
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/sst2")

#2. load tokennizer 
from transformers import AutoTokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#3. tokenize the dataset 
def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

#4. load RoBERTa for classification 
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

c:\Users\Wu\College_App_2028\SciFi_20260616\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5112.41it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
#need to sample down to shorten training time 
import random
# Make sampling reproducible
random.seed(42)

# Total number of training samples
n = len(tokenized["train"])

# Randomly pick 2000 indices
indices = random.sample(range(n), 2000)

# Create the subset
train2000 = tokenized["train"].select(indices)

In [ ]:
"""  
args = TrainingArguments(
    output_dir="roberta-sst2",
    eval_strategy="epoch", 
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
) """


args = TrainingArguments(
    output_dir="outputs",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,

    logging_steps=100,
    dataloader_num_workers=0,

    fp16=False,
    bf16=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train2000, 
    eval_dataset=tokenized["validation"],
)

#7. train
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.379484,0.307219


TrainOutput(global_step=250, training_loss=0.4087329864501953, metrics={'train_runtime': 45117.796, 'train_samples_per_second': 0.044, 'train_steps_per_second': 0.006, 'total_flos': 526222110720000.0, 'train_loss': 0.4087329864501953, 'epoch': 1.0})

In [ ]:
my_metrics = trainer.evaluate()
print("My fine-tuned model:", my_metrics)

Training Loss,Validation Loss,Epoch
0.379484,0.307219,1


My fine-tuned model: {'eval_loss': 0.3072190284729004}


In [ ]:
pred = trainer.predict(tokenized["validation"])

import numpy as np
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
cm

array([[370,  58],
       [ 27, 417]])